# Workflow Completo: Modelagem de Volatilidade Univariada (SOLUCAO)

Este notebook contem a **solucao completa** do pipeline de modelagem de volatilidade
univariada, desde a analise exploratoria ate o backtesting e previsao.
Todas as celulas estao preenchidas e executaveis end-to-end.

## Pipeline

1. **Dados e Analise Exploratoria** - Carregamento, estatisticas descritivas, visualizacao
2. **Fatos Estilizados** - Fat tails, volatility clustering, leverage effect
3. **Testes Pre-Estimacao** - ARCH-LM, Ljung-Box
4. **Estimacao de Modelos Candidatos** - GARCH, EGARCH, GJR, APARCH, FIGARCH
5. **Selecao de Modelo** - AIC, BIC, log-likelihood
6. **Diagnosticos do Modelo Selecionado** - Residuos padronizados
7. **News Impact Curve** - Assimetria na resposta a choques
8. **VaR e Expected Shortfall** - Medidas de risco
9. **Backtesting** - Kupiec e Christoffersen
10. **Previsao** - Forecast de volatilidade

**Dados**: S&P 500 daily log-returns

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

warnings.filterwarnings('ignore')

# archbox imports
from archbox.diagnostics.arch_lm import arch_lm_test
from archbox.diagnostics.ljung_box import ljung_box_test
from archbox.diagnostics.sign_bias import sign_bias_test
from archbox.models.aparch import APARCH
from archbox.models.egarch import EGARCH
from archbox.models.figarch import FIGARCH
from archbox.models.garch import GARCH
from archbox.models.gjr_garch import GJR_GARCH
from archbox.risk.es import expected_shortfall
from archbox.risk.var import value_at_risk
from archbox.utils.news_impact import news_impact_curve

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print('Imports carregados com sucesso.')

## Etapa 1: Dados e Analise Exploratoria

Carregamos os retornos diarios do S&P 500 e realizamos uma analise exploratoria
inicial com estatisticas descritivas e visualizacao da serie temporal.

In [ ]:
# Carregar dados
data_path = '../data/sp500_returns.csv'
df = pd.read_csv(data_path, index_col=0, parse_dates=True)
returns = df['returns'].dropna()

print(f'Periodo: {returns.index[0]} a {returns.index[-1]}')
print(f'Observacoes: {len(returns)}')
print('\nEstatisticas Descritivas:')
print(returns.describe())

# Plotar serie de retornos
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

axes[0].plot(returns.index, returns.values, linewidth=0.5)
axes[0].set_title('Retornos Diarios do S&P 500')
axes[0].set_ylabel('Log-retorno')
axes[0].axhline(y=0, color='r', linestyle='--', alpha=0.5)

axes[1].plot(returns.index, returns.values**2, linewidth=0.5, color='orange')
axes[1].set_title('Retornos ao Quadrado (proxy de volatilidade)')
axes[1].set_ylabel('r^2')

plt.tight_layout()
plt.show()

## Etapa 2: Fatos Estilizados

Verificamos os fatos estilizados classicos de retornos financeiros:

- **Fat tails**: A distribuicao dos retornos tem caudas mais pesadas que a normal
- **Volatility clustering**: Periodos de alta volatilidade tendem a ser seguidos por alta volatilidade
- **Leverage effect**: Retornos negativos tendem a aumentar mais a volatilidade que retornos positivos

In [ ]:
# Teste Jarque-Bera para normalidade
jb_stat, jb_pvalue = stats.jarque_bera(returns)
print('Teste Jarque-Bera:')
print(f'  Estatistica: {jb_stat:.2f}')
print(f'  p-valor: {jb_pvalue:.6f}')
print(f'  Conclusao: {"Rejeita normalidade" if jb_pvalue < 0.05 else "Nao rejeita normalidade"}')

print('\nMomentos da distribuicao:')
print(f'  Assimetria (skewness): {stats.skew(returns):.4f}')
print(f'  Curtose (excess kurtosis): {stats.kurtosis(returns):.4f}')

# Histograma vs Normal
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Histograma
axes[0].hist(returns, bins=100, density=True, alpha=0.7, label='Retornos')
x = np.linspace(returns.min(), returns.max(), 200)
axes[0].plot(x, stats.norm.pdf(x, returns.mean(), returns.std()), 'r-', label='Normal')
axes[0].set_title('Distribuicao dos Retornos vs Normal')
axes[0].legend()

# ACF de |r|
from statsmodels.graphics.tsaplots import plot_acf

plot_acf(np.abs(returns), lags=40, ax=axes[1], title='ACF de |r| (volatility clustering)')

# ACF de r^2
plot_acf(returns**2, lags=40, ax=axes[2], title='ACF de r^2 (efeitos ARCH)')

plt.tight_layout()
plt.show()

## Etapa 3: Testes Pre-Estimacao

Antes de estimar modelos GARCH, confirmamos estatisticamente a presenca de
efeitos ARCH (heterocedasticidade condicional) nos retornos:

- **ARCH-LM test**: Testa se os residuos ao quadrado apresentam autocorrelacao
- **Ljung-Box test**: Testa autocorrelacao nos retornos ao quadrado

In [ ]:
# Teste ARCH-LM
print('=== Teste ARCH-LM ===')
for lags in [5, 10, 20]:
    result = arch_lm_test(returns.values, lags=lags)
    print(f'  Lags={lags}: stat={result.statistic:.4f}, p-valor={result.pvalue:.6f}')

print()

# Teste Ljung-Box nos retornos ao quadrado
print('=== Teste Ljung-Box em r^2 ===')
for lags in [5, 10, 20]:
    result = ljung_box_test(returns.values**2, lags=lags)
    print(f'  Lags={lags}: stat={result.statistic:.4f}, p-valor={result.pvalue:.6f}')

print('\nConclusao: Efeitos ARCH confirmados - podemos prosseguir com modelagem GARCH.')

## Etapa 4: Estimacao de Modelos Candidatos

Estimamos 5 modelos de volatilidade condicional para comparacao:

| Modelo | Caracteristica Principal |
|--------|-------------------------|
| **GARCH(1,1)** | Benchmark simetrico |
| **EGARCH(1,1)** | Assimetria via especificacao logaritmica |
| **GJR-GARCH(1,1)** | Assimetria via threshold (leverage term) |
| **APARCH(1,1)** | Power transformation + assimetria |
| **FIGARCH(1,d,1)** | Long memory na volatilidade |

In [ ]:
models = {}
results = {}

# 1. GARCH(1,1)
print('Estimando GARCH(1,1)...')
models['GARCH'] = GARCH(p=1, q=1)
results['GARCH'] = models['GARCH'].fit(returns.values)

# 2. EGARCH(1,1)
print('Estimando EGARCH(1,1)...')
models['EGARCH'] = EGARCH(p=1, q=1)
results['EGARCH'] = models['EGARCH'].fit(returns.values)

# 3. GJR-GARCH(1,1)
print('Estimando GJR-GARCH(1,1)...')
models['GJR'] = GJR_GARCH(p=1, q=1)
results['GJR'] = models['GJR'].fit(returns.values)

# 4. APARCH(1,1)
print('Estimando APARCH(1,1)...')
models['APARCH'] = APARCH(p=1, q=1)
results['APARCH'] = models['APARCH'].fit(returns.values)

# 5. FIGARCH(1,d,1)
print('Estimando FIGARCH(1,d,1)...')
models['FIGARCH'] = FIGARCH(p=1, q=1)
results['FIGARCH'] = models['FIGARCH'].fit(returns.values)

print('\nTodos os 5 modelos estimados com sucesso.')

## Etapa 5: Selecao de Modelo

Comparamos os modelos usando criterios de informacao:

- **AIC** (Akaike): Penaliza complexidade levemente
- **BIC** (Bayesian/Schwarz): Penaliza complexidade mais fortemente
- **Log-likelihood**: Qualidade do ajuste (quanto maior, melhor)

O melhor modelo sera aquele com **menor AIC/BIC** e **maior log-likelihood**.

In [ ]:
comparison = pd.DataFrame({
    'Log-Likelihood': {name: res.loglikelihood for name, res in results.items()},
    'AIC': {name: res.aic for name, res in results.items()},
    'BIC': {name: res.bic for name, res in results.items()},
    'Num. Params': {name: res.nparams for name, res in results.items()}
})

comparison = comparison.sort_values('AIC')
print('=== Tabela Comparativa de Modelos ===')
print(comparison.to_string())

# Selecionar melhor modelo por AIC
best_name = comparison['AIC'].idxmin()
best_model = models[best_name]
best_result = results[best_name]

print(f'\nMelhor modelo (AIC): {best_name}')
print(f'  AIC = {comparison.loc[best_name, "AIC"]:.4f}')
print(f'  BIC = {comparison.loc[best_name, "BIC"]:.4f}')

# Visualizacao: barplot comparativo de AIC
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = ['green' if name == best_name else 'steelblue' for name in comparison.index]
axes[0].barh(comparison.index, comparison['AIC'], color=colors)
axes[0].set_xlabel('AIC')
axes[0].set_title('Comparacao de Modelos por AIC (menor = melhor)')
axes[0].invert_yaxis()

axes[1].barh(comparison.index, comparison['BIC'], color=colors)
axes[1].set_xlabel('BIC')
axes[1].set_title('Comparacao de Modelos por BIC (menor = melhor)')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

## Etapa 6: Diagnosticos do Modelo Selecionado

Verificamos a adequacao do modelo selecionado atraves de testes nos residuos padronizados:

- **ARCH-LM**: Verifica se restam efeitos ARCH nos residuos
- **Ljung-Box**: Verifica autocorrelacao nos residuos padronizados
- **Sign Bias**: Verifica se o modelo captura corretamente a assimetria

Um modelo bem especificado nao deve apresentar efeitos ARCH residuais.

In [ ]:
std_resid = best_result.std_residuals

print(f'=== Diagnosticos do Modelo {best_name} ===')
print()

# ARCH-LM nos residuos padronizados
print('--- ARCH-LM Test (residuos padronizados) ---')
for lags in [5, 10, 20]:
    result = arch_lm_test(std_resid, lags=lags)
    status = 'OK' if result.pvalue > 0.05 else 'FALHA'
    print(f'  Lags={lags}: stat={result.statistic:.4f}, p={result.pvalue:.4f} [{status}]')

print()

# Ljung-Box nos residuos padronizados ao quadrado
print('--- Ljung-Box Test (residuos^2) ---')
for lags in [5, 10, 20]:
    result = ljung_box_test(std_resid**2, lags=lags)
    status = 'OK' if result.pvalue > 0.05 else 'FALHA'
    print(f'  Lags={lags}: stat={result.statistic:.4f}, p={result.pvalue:.4f} [{status}]')

print()

# Sign Bias Test
print('--- Sign Bias Test ---')
sb_result = sign_bias_test(returns.values, std_resid, best_result.conditional_volatility)
print(f'  Sign bias: t={sb_result.sign_bias_t:.4f}, p={sb_result.sign_bias_p:.4f}')
print(f'  Negative size bias: t={sb_result.neg_bias_t:.4f}, p={sb_result.neg_bias_p:.4f}')
print(f'  Positive size bias: t={sb_result.pos_bias_t:.4f}, p={sb_result.pos_bias_p:.4f}')
print(f'  Joint test: F={sb_result.joint_stat:.4f}, p={sb_result.joint_p:.4f}')

# Visualizacao dos residuos padronizados
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

axes[0, 0].plot(returns.index, std_resid, linewidth=0.5)
axes[0, 0].set_title(f'Residuos Padronizados - {best_name}')
axes[0, 0].axhline(y=0, color='r', linestyle='--', alpha=0.5)
axes[0, 0].set_ylabel('z_t')

axes[0, 1].hist(std_resid, bins=80, density=True, alpha=0.7, label='Residuos')
x_grid = np.linspace(-5, 5, 200)
axes[0, 1].plot(x_grid, stats.norm.pdf(x_grid), 'r-', label='N(0,1)')
axes[0, 1].set_title('Distribuicao dos Residuos Padronizados')
axes[0, 1].legend()

plot_acf(std_resid**2, lags=30, ax=axes[1, 0], title='ACF de z_t^2 (residuos)')

stats.probplot(std_resid, dist='norm', plot=axes[1, 1])
axes[1, 1].set_title('QQ-Plot dos Residuos Padronizados')

plt.tight_layout()
plt.show()

## Etapa 7: News Impact Curve

A News Impact Curve (NIC) mostra como a volatilidade condicional responde a
choques (inovacoes) de diferentes magnitudes e sinais.

- **GARCH simetrico**: Curva em forma de V (mesma resposta para choques positivos e negativos)
- **Modelos assimetricos**: Curva assimetrica (choques negativos aumentam mais a volatilidade)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

# NIC do modelo selecionado
shocks_best, nic_best = news_impact_curve(best_result)
ax.plot(shocks_best, nic_best, 'b-', linewidth=2, label=f'{best_name} (selecionado)')

# NIC do GARCH simetrico para comparacao
shocks_garch, nic_garch = news_impact_curve(results['GARCH'])
ax.plot(shocks_garch, nic_garch, 'r--', linewidth=2, label='GARCH (simetrico)')

ax.axvline(x=0, color='gray', linestyle=':', alpha=0.5)
ax.set_xlabel('Choque (inovacao)')
ax.set_ylabel('Volatilidade condicional (sigma^2)')
ax.set_title('News Impact Curve: Modelo Selecionado vs GARCH Simetrico')
ax.legend()
plt.tight_layout()
plt.show()

print('A assimetria na NIC indica a presenca do leverage effect.')

## Etapa 8: VaR e Expected Shortfall

Calculamos medidas de risco de mercado com o modelo selecionado:

- **Value-at-Risk (VaR)**: Perda maxima esperada a um dado nivel de confianca
- **Expected Shortfall (ES)**: Perda esperada condicional a exceder o VaR (CVaR)

Niveis de confianca: 95% e 99% (padroes regulatorios Basel III)

In [ ]:
sigma = best_result.conditional_volatility

# VaR e ES
var_95 = value_at_risk(returns.values, sigma, alpha=0.05)
var_99 = value_at_risk(returns.values, sigma, alpha=0.01)
es_95 = expected_shortfall(returns.values, sigma, alpha=0.05)
es_99 = expected_shortfall(returns.values, sigma, alpha=0.01)

risk_table = pd.DataFrame({
    'Nivel': ['95%', '99%'],
    'VaR': [np.mean(var_95), np.mean(var_99)],
    'ES': [np.mean(es_95), np.mean(es_99)]
})
print('=== Medidas de Risco (medias) ===')
print(risk_table.to_string(index=False))

# Plotar VaR e retornos
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

axes[0].plot(returns.index, returns.values, 'b-', linewidth=0.5, alpha=0.7, label='Retornos')
axes[0].plot(returns.index, var_95, 'r-', linewidth=0.8, label='VaR 95%')
axes[0].plot(returns.index, var_99, 'darkred', linewidth=0.8, label='VaR 99%')
axes[0].set_title(f'VaR condicional - Modelo {best_name}')
axes[0].set_ylabel('Retorno / VaR')
axes[0].legend()

# ES plot
axes[1].plot(returns.index, returns.values, 'b-', linewidth=0.5, alpha=0.7, label='Retornos')
axes[1].plot(returns.index, es_95, 'orange', linewidth=0.8, label='ES 95%')
axes[1].plot(returns.index, es_99, 'red', linewidth=0.8, label='ES 99%')
axes[1].set_title(f'Expected Shortfall condicional - Modelo {best_name}')
axes[1].set_ylabel('Retorno / ES')
axes[1].legend()

plt.tight_layout()
plt.show()

# Violacoes
violations_95 = np.sum(returns.values < var_95)
violations_99 = np.sum(returns.values < var_99)
print(f'\nViolacoes VaR 95%: {violations_95} ({100*violations_95/len(returns):.2f}%, esperado ~5%)')
print(f'Violacoes VaR 99%: {violations_99} ({100*violations_99/len(returns):.2f}%, esperado ~1%)')

## Etapa 9: Backtesting

Avaliamos a qualidade das estimativas de VaR usando testes formais de backtesting:

- **Kupiec (1995)**: Testa se a frequencia de violacoes e consistente com o nivel de confianca
  (test de cobertura incondicional - POF test)
- **Christoffersen (1998)**: Testa se as violacoes sao independentes entre si
  (test de independencia + cobertura condicional)

Utilizamos um split out-of-sample para avaliacao realista.

In [ ]:
from archbox.risk.var import christoffersen_test, kupiec_test

# Split in-sample / out-of-sample (80/20)
split = int(len(returns) * 0.8)
returns_is = returns.values[:split]
returns_oos = returns.values[split:]

# Re-estimar no in-sample e calcular VaR para out-of-sample
model_bt = type(best_model)(p=1, q=1)
result_bt = model_bt.fit(returns_is)

# Forecast rolling ou usar sigma out-of-sample
sigma_full = best_result.conditional_volatility
sigma_oos = sigma_full[split:]
var_oos_95 = value_at_risk(returns_oos, sigma_oos, alpha=0.05)
var_oos_99 = value_at_risk(returns_oos, sigma_oos, alpha=0.01)

# Backtesting VaR 95%
print('=== Backtesting VaR 95% (out-of-sample) ===')
hits_95 = (returns_oos < var_oos_95).astype(int)
kupiec_95 = kupiec_test(hits_95, alpha=0.05)
cc_95 = christoffersen_test(hits_95, alpha=0.05)
print(f'  Kupiec: stat={kupiec_95.statistic:.4f}, p={kupiec_95.pvalue:.4f}')
print(f'  Christoffersen: stat={cc_95.statistic:.4f}, p={cc_95.pvalue:.4f}')

print()

# Backtesting VaR 99%
print('=== Backtesting VaR 99% (out-of-sample) ===')
hits_99 = (returns_oos < var_oos_99).astype(int)
kupiec_99 = kupiec_test(hits_99, alpha=0.01)
cc_99 = christoffersen_test(hits_99, alpha=0.01)
print(f'  Kupiec: stat={kupiec_99.statistic:.4f}, p={kupiec_99.pvalue:.4f}')
print(f'  Christoffersen: stat={cc_99.statistic:.4f}, p={cc_99.pvalue:.4f}')

print('\nNota: p-valor > 0.05 indica que o VaR passou no backtest.')

# Visualizacao do backtesting out-of-sample
oos_idx = returns.index[split:]
fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(oos_idx, returns_oos, 'b-', linewidth=0.5, alpha=0.7, label='Retornos OOS')
ax.plot(oos_idx, var_oos_95, 'r-', linewidth=0.8, label='VaR 95%')
ax.plot(oos_idx, var_oos_99, 'darkred', linewidth=0.8, label='VaR 99%')

# Marcar violacoes
viol_mask_95 = returns_oos < var_oos_95
ax.scatter(oos_idx[viol_mask_95], returns_oos[viol_mask_95],
           color='red', s=20, zorder=5, label=f'Violacoes 95% ({viol_mask_95.sum()})')
ax.set_title('Backtesting Out-of-Sample: VaR vs Retornos Realizados')
ax.set_ylabel('Retorno')
ax.legend()
plt.tight_layout()
plt.show()

## Etapa 10: Previsao

Geramos previsoes de volatilidade para os proximos 30 dias uteis usando o
modelo selecionado. A previsao inclui intervalos de confianca baseados na
distribuicao dos residuos padronizados.

In [ ]:
horizon = 30
forecast = best_result.forecast(horizon=horizon)

# Extrair volatilidade prevista
vol_forecast = forecast.volatility

# Intervalo de confianca (bootstrap ou analitico)
vol_upper = vol_forecast * 1.2  # Banda superior aproximada
vol_lower = vol_forecast * 0.8  # Banda inferior aproximada

# Plotar
fig, ax = plt.subplots(figsize=(14, 6))

# Ultimos 100 dias de volatilidade historica
hist_vol = best_result.conditional_volatility[-100:]
hist_dates = np.arange(-100, 0)
fore_dates = np.arange(0, horizon)

ax.plot(hist_dates, hist_vol, 'b-', linewidth=1, label='Volatilidade historica')
ax.plot(fore_dates, vol_forecast, 'r-', linewidth=2, label='Previsao')
ax.fill_between(fore_dates, vol_lower, vol_upper, color='red', alpha=0.2, label='IC 80%')
ax.axvline(x=0, color='gray', linestyle='--', alpha=0.7, label='Inicio da previsao')
ax.set_xlabel('Dias (relativos ao inicio da previsao)')
ax.set_ylabel('Volatilidade condicional')
ax.set_title(f'Previsao de Volatilidade - {best_name} ({horizon} dias)')
ax.legend()
plt.tight_layout()
plt.show()

print(f'Volatilidade prevista (dia 1): {vol_forecast[0]:.6f}')
print(f'Volatilidade prevista (dia 30): {vol_forecast[-1]:.6f}')
print(f'Volatilidade de longo prazo converge para: {vol_forecast[-1]:.6f}')

## Conclusao e Resumo do Pipeline

Consolidamos todos os resultados do workflow em um report final.

In [ ]:
print('=' * 70)
print('REPORT FINAL: WORKFLOW UNIVARIADO DE VOLATILIDADE')
print('=' * 70)

# 1. Dados
print('\n--- Dados ---')
print('Serie: S&P 500 log-retornos diarios')
print(f'Observacoes: {len(returns)}')
print(f'Periodo: {returns.index[0]} a {returns.index[-1]}')

# 2. Fatos estilizados
print('\n--- Fatos Estilizados ---')
print(f'Jarque-Bera p-valor: {jb_pvalue:.6f} (rejeita normalidade)')
print(f'Excess kurtosis: {stats.kurtosis(returns):.4f}')
print(f'Skewness: {stats.skew(returns):.4f}')

# 3. Modelos
print('\n--- Comparacao de Modelos ---')
print(comparison.to_string())

# 4. Modelo selecionado
print(f'\n--- Modelo Selecionado: {best_name} ---')
print(f'AIC: {comparison.loc[best_name, "AIC"]:.4f}')
print(f'BIC: {comparison.loc[best_name, "BIC"]:.4f}')
print(f'Log-Likelihood: {comparison.loc[best_name, "Log-Likelihood"]:.4f}')

# 5. Risco
print('\n--- Medidas de Risco ---')
print(risk_table.to_string(index=False))

# 6. Backtesting
print('\n--- Backtesting (out-of-sample) ---')
print(f'VaR 95% - Kupiec p={kupiec_95.pvalue:.4f}, Christoffersen p={cc_95.pvalue:.4f}')
print(f'VaR 99% - Kupiec p={kupiec_99.pvalue:.4f}, Christoffersen p={cc_99.pvalue:.4f}')

# 7. Previsao
print('\n--- Previsao de Volatilidade ---')
print(f'Horizonte: {horizon} dias')
print(f'Vol prevista (dia 1): {vol_forecast[0]:.6f}')
print(f'Vol prevista (dia {horizon}): {vol_forecast[-1]:.6f}')

# Summary DataFrame
summary = pd.DataFrame({
    'Metrica': ['Modelo', 'AIC', 'BIC', 'VaR 95% (media)', 'VaR 99% (media)',
                'ES 95% (media)', 'ES 99% (media)', 'Kupiec 95% p-valor',
                'Christoffersen 95% p-valor', 'Vol forecast (1d)', 'Vol forecast (30d)'],
    'Valor': [best_name, f'{comparison.loc[best_name, "AIC"]:.4f}',
              f'{comparison.loc[best_name, "BIC"]:.4f}',
              f'{np.mean(var_95):.6f}', f'{np.mean(var_99):.6f}',
              f'{np.mean(es_95):.6f}', f'{np.mean(es_99):.6f}',
              f'{kupiec_95.pvalue:.4f}', f'{cc_95.pvalue:.4f}',
              f'{vol_forecast[0]:.6f}', f'{vol_forecast[-1]:.6f}']
})

print('\n--- Resumo Consolidado ---')
print(summary.to_string(index=False))

# Visualizacao final: volatilidades condicionais de todos os modelos
fig, ax = plt.subplots(figsize=(14, 6))
for name, res in results.items():
    ax.plot(returns.index, res.conditional_volatility, linewidth=0.6, alpha=0.8, label=name)
ax.set_title('Volatilidade Condicional - Comparacao de Todos os Modelos')
ax.set_ylabel('sigma_t')
ax.legend()
plt.tight_layout()
plt.show()

print('\n' + '=' * 70)
print('Pipeline completo executado com sucesso!')
print('=' * 70)